# CoverageVision AI: train the U-Net verifier (B1) on a GPU

Works on **Google Colab** (T4/L4/A100) and on a generic **Jupyter GPU node** (for example NCSA with an H100).
Steps: set up and clone the private repo, install, generate the synthetic dataset, train, evaluate, zip the results.
Full instructions: `Docs/gpu_training.md`. Secrets are read from an environment variable or `getpass` at run time and
are never stored in this notebook.

In [ ]:
# 1. Environment and working directory
import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
if IN_COLAB:
    WORKDIR = "/content"
else:
    # NCSA / other Jupyter: set this to a fast, persistent directory with enough space (a few GB), for example
    # "/projects/<allocation>/<user>/coverageai" or "$WORK/coverageai". Defaults to $WORKDIR, then $HOME.
    WORKDIR = os.environ.get("WORKDIR") or os.path.expanduser("~")
WORKDIR = pathlib.Path(WORKDIR).expanduser()
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
REPO_URL = "https://github.com/MasterKinjalk/coverageai.git"
BRANCH = "main"                       # use the branch that has vision/unet.py and vision/render.py
REPO_DIR = WORKDIR / "coverageai"
REPO_ZIP = None                       # fallback: path to an uploaded zip of the repo, e.g. WORKDIR / "coverageai.zip"
print("colab:", IN_COLAB, "| workdir:", WORKDIR, "| cpus:", os.cpu_count())

def sh(cmd, **kw):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    return subprocess.run(cmd, shell=isinstance(cmd, str), check=True, **kw)

In [ ]:
# 2. Get the code. Token: env var GITHUB_TOKEN, else a hidden prompt. Use a fine-grained, read-only token.
import getpass, shutil, zipfile
if not (REPO_DIR / "pyproject.toml").exists():
    if REPO_ZIP and pathlib.Path(REPO_ZIP).exists():                      # fallback: uploaded zip
        with zipfile.ZipFile(REPO_ZIP) as z:
            z.extractall(WORKDIR / "_unzip")
        inner = next(p for p in (WORKDIR / "_unzip").rglob("pyproject.toml")).parent
        shutil.move(str(inner), str(REPO_DIR))
    else:
        token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token (read-only, not stored): ")
        auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
        try:
            print("$ git clone --depth 1 --branch", BRANCH, REPO_URL, "(token hidden)")
            r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, auth_url, str(REPO_DIR)],
                               capture_output=True, text=True)   # not via sh(): it would print the token
            if r.returncode != 0:   # report without the token (a CalledProcessError would show the URL)
                raise RuntimeError("git clone failed: " + r.stderr.replace(token, "***")) from None
        finally:
            del token, auth_url
        sh(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])   # drop the token from .git/config
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
os.environ["PYTHONPATH"] = str(REPO_DIR / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
print("repo at", REPO_DIR)

In [ ]:
# 3. Install. Keep a CUDA torch if the image already has one (Colab, NCSA); otherwise install the [vision] extra.
def cuda_torch_ok():
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False

pip = [sys.executable, "-m", "pip", "install", "-q"]
extras = ["segmentation-models-pytorch", "opencv-python-headless", "scikit-image", "scipy", "pandas", "pyarrow",
          "pyyaml", "matplotlib", "requests", "timm"]
if cuda_torch_ok():
    sh(pip + extras)                                          # torch stays as shipped
    try:
        sh(pip + ["-e", ".", "--no-deps"])
    except subprocess.CalledProcessError:                     # e.g. kernel Python older than requires-python
        print("editable install failed; using PYTHONPATH=src instead")
else:
    try:
        sh(pip + ["-e", ".[vision]"])
    except subprocess.CalledProcessError:
        sh(pip + ["torch", "torchvision"] + extras)
import importlib; importlib.invalidate_caches()
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())
else:
    print("WARNING: no GPU. Training will run on CPU (slow). In Colab: Runtime > Change runtime type > GPU.")

In [ ]:
# 4. Generate the synthetic dataset on this machine (all CPU cores). More data: N_CARDS = 3000.
N_CARDS = 1200
DATA = REPO_DIR / "data" / "synthetic" / "v1"
if not (DATA / "splits.json").exists():
    sh([sys.executable, "-m", "coverageai.vision.render", "--out", str(DATA), "--n", str(N_CARDS), "--seed", "0"])
print(open(DATA / "summary.json").read()[:600])

## 5. Train
`bf16` autocast on Ampere/Hopper GPUs (H100, A100, L4), `fp16` with a grad scaler on older ones (T4); `amp="auto"` picks.
Batch 48 at crop 512 fits an H100; on a T4 use `batch=16, crop=320`. A checkpoint is written every epoch
(`models/unet_v1.last.pt`) and the best-val-Dice one to `models/unet_v1.pt`.

In [ ]:
from coverageai.vision import unet
gpu = torch.cuda.is_available()
cfg = dict(
    device="cuda" if gpu else "cpu",
    amp="auto",
    batch=48 if gpu else 8,          # T4: 16
    crop=512 if gpu else 256,        # T4: 320
    workers=min(os.cpu_count() or 2, 16) if gpu else 0,
    max_minutes=45,
    epochs=40,
)
res = unet.train(DATA, out="models/unet_v1.pt", epochs=cfg["epochs"], batch=cfg["batch"], crop=cfg["crop"],
                 max_minutes=cfg["max_minutes"], device=cfg["device"], seed=0, amp=cfg["amp"],
                 num_workers=cfg["workers"], crops_per_card=6, max_steps_per_epoch=300,
                 patience=10, log_path="results/unet_train_log.csv")
res

## 6. Evaluate on the test split
Writes `results/b1_synthetic.json` and `experiments/figures/b1_parity.png`.

In [ ]:
import json, pandas as pd
r = unet.evaluate(DATA, "test", "models/unet_v1.pt", "results/b1_synthetic.json",
                  "experiments/figures/b1_parity.png", device=cfg["device"], verbose=False)
o = r["overall"]
print(f"coverage MAE {o['coverage_mae_pp']:.2f} pp (target <= 2) | IoU {o['iou']:.3f} | "
      f"class agreement {o['class_agreement']:.3f} (target >= 0.90) | n={o['n']}")
pd.DataFrame(r["by_band"]).T

In [ ]:
from IPython.display import Image, display
display(Image("experiments/figures/b1_parity.png"))
pd.read_csv("results/unet_train_log.csv").tail()

In [ ]:
# 7. Zip the artefacts for download.
import zipfile
OUT_ZIP = WORKDIR / "b1_unet_results.zip"
files = ["models/unet_v1.pt", "results/b1_synthetic.json", "results/unet_train_log.csv",
         "experiments/figures/b1_parity.png"]
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if os.path.exists(f):
            z.write(f)
print("wrote", OUT_ZIP, f"({OUT_ZIP.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(str(OUT_ZIP))
else:
    print("Download this file from the Jupyter file browser (or scp it):", OUT_ZIP)